In [1]:
#pip install torch torchvision torchaudio

In [ ]:
import torch
from torch import nn
from collections import Counter

In [3]:
# setup for my m1pro laptop
if torch.backends.mps.is_available():
    device = torch.device("mps") # Apple Silicon GPU
elif torch.cuda.is_available():
    device = torch.device("cuda") # NVIDIA GPU - if available
else:
    device = torch.device("cpu")

print("Using device:", device)

Using device: mps


In [4]:
class BrainTumorCNN(nn.Module):
    """
    Improved CNN for brain tumor classification.

    Key differences vs old SimpleCNN:
    - Multi-scale first block: 3x3 and 5x5 conv branches in parallel.
    - Deeper conv blocks so receptive field covers larger brain regions (better for asymmetry).
    - Texture-focused final block with more channels.
    - Global Average Pooling instead of huge Flatten+Linear (fewer params).
    - Still lightweight enough to train on an M1 Pro chip.
    """
    def __init__(self, in_channels: int = 3, num_classes: int = 1):
        # in_channels: number of input channels (3 for RGB)
        # num_classes: number of output classes (1 for binary classification)
        super().__init__()

        # -------- Block 1: Multi-scale feature extraction --------
        # 3x3 conv ( edges, boundaries)
        # 5x5 conv ( larger structures, coarse tumor shape)
        # 2 branches in parallel to capture different scales of features
        self.branch3x3 = nn.Sequential(
            nn.Conv2d(in_channels, 16, kernel_size=3, padding=1),
            nn.ReLU()
        )
        # Branch B: 5x5 conv (larger structures, coarse tumor shape)
        self.branch5x5 = nn.Sequential(
            nn.Conv2d(in_channels, 16, kernel_size=5, padding=2),
            nn.ReLU()
        )

        # After concatenation, channels = 16 + 16 = 32
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)  # 64x64 -> 32x32

        # -------- Block 2: Deeper receptive field --------
        # Two stacked 3x3 convs increase effective receptive field, which helps capture asymmetry or larger patterns of tumors
        self.block2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2) # 32x32 -> 16x16
        )

        # -------- Block 3: Texture-focused block --------
        # More channels, still 3x3, no more spatial downsampling
        # Designed to learn tumor texture (heterogeneity, contrast) which is the overall appearance
        self.block3 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.ReLU()
        )

        # -------- Global Average Pooling + Classifier --------
        # Global Average Pooling: each channel -> 1 scalar (how strong that pattern is anywhere)
        # benefits: fewer parameters, less overfitting, spatial invariance
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))  # (N, C, H, W) -> (N, C, 1, 1)

        self.classifier = nn.Sequential(
            nn.Flatten(),               # (N, 128, 1, 1) -> (N, 128)
            nn.Dropout(p=0.5),          # Regularization (important for small-ish dataset)
            nn.Linear(128, 1)           # Single logit for BCEWithLogitsLoss
        )

    def forward(self, x):
        # x: (N, 3, 64, 64)

        # Multi-scale block
        x3 = self.branch3x3(x)        
        x5 = self.branch5x5(x)        
        x = torch.cat([x3, x5], dim=1)

        x = self.pool1(x)             

        # run both branches and concatenate
        # Deeper block
        x = self.block2(x)            

        # Texture block
        x = self.block3(x)            

        # Global average pooling
        x = self.global_pool(x)       

        # Classifier
        x = self.classifier(x) # (N, 1)

        return x

In [5]:
model = BrainTumorCNN(in_channels=3, num_classes=1).to(device)
print(model)

BrainTumorCNN(
  (branch3x3): Sequential(
    (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
  )
  (branch5x5): Sequential(
    (0): Conv2d(3, 16, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
    (1): ReLU()
  )
  (pool1): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (block2): Sequential(
    (0): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU()
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (block3): Sequential(
    (0): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU()
  )
  (global_pool): AdaptiveAvgPool2d(output_size=(1, 1))
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Dropout(p=0.5, i

In [6]:
from torchvision import transforms, datasets
from torch.utils.data import DataLoader
import time

In [7]:
data_path = "../dataset_with_label"

In [ ]:
# -------- Transforms --------
train_transform = transforms.Compose([
    transforms.Resize((64, 64)), # resize to 64x64 so model input size matches
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.1, contrast=0.1), # color jitter for brightness and contrast
    transforms.ToTensor(), # convert PIL image to Tensor
    transforms.Normalize(mean=[0.5]*3, std=[0.5]*3) # dataset normalization
])

test_transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5]*3, std=[0.5]*3)
])

# ---- Load ImageFolder datasets ----
train_data = datasets.ImageFolder(root=f"{data_path}/train", transform=train_transform)
test_data  = datasets.ImageFolder(root=f"{data_path}/test",  transform=test_transform)

print("Train samples:", len(train_data))
print("Test samples:", len(test_data))

train_dataloader = DataLoader(train_data, batch_size=16, shuffle=True)
test_dataloader  = DataLoader(test_data,  batch_size=16, shuffle=False)

print("Dataloaders ready.")

Train samples: 3009
Test samples: 753
Dataloaders ready.


In [10]:
# the formula for dice coefficient is 2 * (|X ∩ Y|) / (|X| + |Y|), X is the predicted set, Y is the ground truth set
# in confusion matrix terms, dice = 2TP / (2TP + FP + FN), TP: # of images correctly predicted as positive,
# FP: preddicted 1, then actually 0, FN: predicted 0, then actually 1
# dice coefficient is a measure of overlap between two samples, focusing on the positive class(tumor presence)

def dice_coefficient(logits, targets, threshold=0.5, eps=0.00001):
    """
    Compute Dice coefficient for binary classification (tumor = positive class).

    logits: model outputs (before sigmoid), shape (batch,)
    targets: true labels (0 or 1), shape (batch,)
    threshold: probability threshold to decide positive vs negative
    eps: small constant to avoid division by zero
    """
    probs = torch.sigmoid(logits)
    preds = (probs > threshold).float()
    targets = targets.float()

    # intersection = number of true positives
    intersection = (preds * targets).sum()

    # sum of predicted positives + true positives
    union = preds.sum() + targets.sum()

    # if both preds and targets are all zeros, union = 0; eps keeps it stable
    dice = (2 * intersection + eps) / (union + eps)

    return dice

In [11]:
# ----- Single epoch training -----
def train_step(model, dataloader, loss_fn, optimizer, device):
    ''' Performs a single epoch training step 
    arguments:
    model: the neural network model
    dataloader: DataLoader for training data
    loss_fn: loss function
    optimizer: optimizer for updating model parameters
    device: computation device (cpu, cuda, mps)
    returns:
    average loss, average accuracy, time taken for the epoch'''
    model.train() # this is for training mode
    total_loss, total_acc, total_dice = 0, 0, 0
    start = time.time()

    for X, y in dataloader:
        X, y = X.to(device), y.to(device).float() # mps and cuda need float labels for BCEWithLogitsLoss

        logits = model(X).squeeze(1)
        loss = loss_fn(logits, y)

        # update model parameters by backpropagation
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        # logits to predictions, when sigmoid>0.5 -> class 1 else class 0
        preds = (torch.sigmoid(logits) > 0.5).int()
        total_acc += (preds == y.int()).sum().item() / len(y)

        # dice
        batch_dice = dice_coefficient(logits, y)
        total_dice += batch_dice.item()

    num_batches = len(dataloader)
    return (
        total_loss / num_batches,
        total_acc / num_batches,
        total_dice / num_batches,
        time.time() - start
    )


# ----- Single epoch testing -----
def test_step(model, dataloader, loss_fn, device):
    ''' Performs a single epoch testing/validation step
    arguments:
    model: the neural network model
    dataloader: DataLoader for testing/validation data
    loss_fn: loss function
    device: computation device (cpu, cuda, mps)
    returns:
    average loss, average accuracy, time taken for the epoch'''
    model.eval()
    total_loss, total_acc, total_dice = 0, 0, 0
    start = time.time()

    with torch.inference_mode():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device).float()
            logits = model(X).squeeze(1)
            loss = loss_fn(logits, y)

            total_loss += loss.item()
            preds = (torch.sigmoid(logits) > 0.5).int()
            total_acc += (preds == y.int()).sum().item() / len(y)

            batch_dice = dice_coefficient(logits, y)
            total_dice += batch_dice.item()

    num_batches = len(dataloader)
    return (
        total_loss / num_batches,
        total_acc / num_batches,
        total_dice / num_batches,
        time.time() - start
    )


# ----- Train Loop -----
def train(model, train_dataloader, test_dataloader, optimizer, loss_fn, device, epochs=10):
    ''' Full training loop over multiple epochs, combines train_step and test_step 
    Loss(BCEWithLogitsLoss) is what the optimizer minimizes '''
    for epoch in range(epochs):
        train_loss, train_acc, train_dice, t_time = train_step(
            model, train_dataloader, loss_fn, optimizer, device
        )
        test_loss, test_acc, test_dice, v_time = test_step(
            model, test_dataloader, loss_fn, device
        )

        print(
            f"Epoch {epoch+1}/{epochs} | "
            f"Train: loss={train_loss:.4f}, acc={train_acc:.4f}, dice={train_dice:.4f}, time={t_time:.2f}s | "
            f"Test: loss={test_loss:.4f}, acc={test_acc:.4f}, dice={test_dice:.4f}, time={v_time:.2f}s"
        )

In [12]:
model = BrainTumorCNN().to(device)

loss_fn = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

results = train(
    model=model,
    train_dataloader=train_dataloader,
    test_dataloader=test_dataloader,
    optimizer=optimizer,
    loss_fn=loss_fn,
    device=device,
    epochs=10
)

Epoch 1/10 | Train: loss=0.6256, acc=0.6429, dice=0.7303, time=5.87s | Test: loss=0.5235, acc=0.8125, dice=0.5705, time=0.62s
Epoch 2/10 | Train: loss=0.4379, acc=0.8148, dice=0.8274, time=4.41s | Test: loss=0.3854, acc=0.8464, dice=0.5282, time=0.56s
Epoch 3/10 | Train: loss=0.3842, acc=0.8380, dice=0.8460, time=4.50s | Test: loss=0.3168, acc=0.8711, dice=0.5239, time=0.60s
Epoch 4/10 | Train: loss=0.3487, acc=0.8575, dice=0.8639, time=4.49s | Test: loss=0.2593, acc=0.8958, dice=0.5811, time=0.58s
Epoch 5/10 | Train: loss=0.2789, acc=0.8929, dice=0.8977, time=4.47s | Test: loss=0.2378, acc=0.9141, dice=0.6091, time=0.54s
Epoch 6/10 | Train: loss=0.2394, acc=0.9081, dice=0.9142, time=4.33s | Test: loss=0.2187, acc=0.9206, dice=0.6823, time=0.60s
Epoch 7/10 | Train: loss=0.2008, acc=0.9302, dice=0.9338, time=4.39s | Test: loss=0.1848, acc=0.9401, dice=0.6340, time=0.55s
Epoch 8/10 | Train: loss=0.1742, acc=0.9362, dice=0.9417, time=4.37s | Test: loss=0.1561, acc=0.9427, dice=0.6007, tim

# Explain why dice is much lower than acc:

Loss decreases and both training accuracy and Dice rise toward ~0.94, meaning the classifier becomes very confident and consistent on the training set. On the test set, accuracy also improves from ~0.81 to ~0.94, indicating that the model generalizes well. However, the Dice coefficient on the test set stays much lower (0.52–0.68) than both accuracy and training Dice, which suggests that the model struggles with the positive (tumor) class and is sensitive to false negatives and false positives. 

This could means class imbalance and fewer tumor images, and the model may be leaning toward predicting the majority class. __This is not our case.__

A possible solution is to adjust thresholds below or above 0.5 until the best Dice. Or, decrease the number of batch size.

In [ ]:
# Compare 1(tumor) and 0(no tumor)
if Counter(y for _, y in train_data)[1] >= Counter(y for _, y in train_data)[0]:
    print("Train: More tumor(1) samples than no tumor(0) samples")
else:
    print("Train: More no tumor(0) samples than tumor(1) samples")

Train: More tumor(1) samples than no tumor(0) samples
